# MTH9877 — Assignment 3: Part E Extensions

**Standalone notebook — run independently after `Assignment3.ipynb` has been executed at least once.**

Requires cached files in `processed/`:
- `survival_loans.parquet` — built by Step 1
- `panel_monthly.parquet` — built by Step 3
- `model_dc_weights.pt`, `xgb_model.json`, `scaler_dc.pkl`, `ml_features.json`, `dc_df.parquet`, `dc_meta.npz` — exported by the artifact-export cell at the end of Part D

---

## Part E — Extensions

- **E.1** Competing Risks: Prepayment vs. Default (Aalen-Johansen CIF)
- **E.2** Scenario Analysis: Interest Rate Shocks

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import json as _json
import pickle as _pickle
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
import torch
import torch.nn as nn
import xgboost as xgb
from lifelines import AalenJohansenFitter
from lifelines.utils import concordance_index
warnings.filterwarnings("ignore")

BASE      = Path("/Users/yueqilin/Desktop/MTH9877 IR/IR&C Assignment3")
OUT_DIR   = BASE / "processed"
PANEL_PATH = OUT_DIR / "panel_monthly.parquet"

DEVICE = (
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("cpu")
)
print(f"Device: {DEVICE}")
print(f"processed/ contents: {[p.name for p in sorted(OUT_DIR.iterdir())]}")

/Users/yueqilin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Device: mps
processed/ contents: ['A1_km_hazard.png', 'A2_km_fico.png', 'A2_km_ltv.png', 'A2_km_vintage.png', 'B1_cox_hr.png', 'B2_schoenfeld.png', 'C_backtest.png', 'C_feature_importance.png', 'D_gradient_sensitivity.png', 'D_survival_curves.png', 'D_training_loss.png', 'E1_competing_risks_cif.png', 'E2_scenario_analysis.png', 'macro_monthly.parquet', 'panel_monthly.parquet', 'survival_loans.parquet']


## Setup — Load Cached Artifacts

In [ ]:
# ── Deep Cox model definition (must match main notebook) ─────────────────────
class DeepCox(nn.Module):
    def __init__(self, in_features: int, hidden: list = [128, 64, 32], dropout: float = 0.2):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


# ── Load scalar metadata ──────────────────────────────────────────────────────
meta = np.load(str(OUT_DIR / "dc_meta.npz"), allow_pickle=True)
feat_cols_dc = list(meta["feat_cols_dc"])
n_train      = int(meta["n_train"][0])
print(f"feat_cols_dc : {feat_cols_dc}")
print(f"n_train      : {n_train:,}")

with open(OUT_DIR / "scaler_dc.pkl", "rb") as f:
    scaler_dc = _pickle.load(f)

with open(OUT_DIR / "ml_features.json") as f:
    FEATURES = _json.load(f)
print(f"FEATURES     : {len(FEATURES)} columns")

dc_df = pd.read_parquet(OUT_DIR / "dc_df.parquet")
print(f"dc_df        : {len(dc_df):,} rows")

# ── Load Deep Cox weights ─────────────────────────────────────────────────────
state = torch.load(OUT_DIR / "model_dc_weights.pt", map_location=DEVICE, weights_only=True)
first_layer_key = [k for k in state if 'net.0.weight' in k][0]
in_dim = state[first_layer_key].shape[1]
print(f"model in_dim : {in_dim}")

model_dc = DeepCox(in_dim, hidden=[256, 128, 64], dropout=0.3).to(DEVICE)
model_dc.load_state_dict(state)
model_dc.eval()
print("Deep Cox model loaded")

# ── Load XGBoost ──────────────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(str(OUT_DIR / "xgb_model.json"))
print("XGBoost model loaded")

---
## E.1 Competing Risks: Prepayment vs. Default

Use the **Aalen-Johansen estimator** for cumulative incidence functions (CIF).
Unlike the Kaplan-Meier 1 − S(t) complement, AJ correctly handles competing risks:
treating default as a competing event for prepayment (and vice versa) avoids
over-estimating each CIF by naively censoring the other event.

Event codes: prepayment = 1, default = 2, censored = 0.

In [ ]:
# ── Load survival dataset and build stratified 100K subsample ─────────────────
survival = pl.read_parquet(OUT_DIR / "survival_loans.parquet")
print(f"Full dataset: {survival.height:,} loans")

B_SAMPLE_N = 100_000
sv_sub_pl = (
    survival
    .with_columns(pl.col("VintageYear").cast(pl.Int32))
    .group_by("VintageYear")
    .map_groups(lambda g: g.sample(
        n=min(len(g), max(1, int(B_SAMPLE_N * len(g) / survival.height))),
        seed=42
    ))
)
sv_sub = sv_sub_pl.to_pandas()
print(f"Subsample   : {len(sv_sub):,} loans")

# ── Build event type ──────────────────────────────────────────────────────────
sv_cr = sv_sub[["duration", "prepaid", "defaulted"]].copy()
sv_cr["event_type"] = 0
sv_cr.loc[sv_cr["prepaid"]   == 1, "event_type"] = 1
sv_cr.loc[sv_cr["defaulted"] == 1, "event_type"] = 2

print(f"Prepaid  : {(sv_cr['event_type']==1).sum():,}")
print(f"Defaulted: {(sv_cr['event_type']==2).sum():,}")
print(f"Censored : {(sv_cr['event_type']==0).sum():,}")

T_cr = sv_cr["duration"]
E_cr = sv_cr["event_type"]

fig, ax = plt.subplots(figsize=(9, 5))

ajf_prepay = AalenJohansenFitter()
ajf_prepay.fit(T_cr, E_cr, event_of_interest=1, label="Prepayment")
ajf_prepay.plot_cumulative_density(ax=ax)

ajf_default = AalenJohansenFitter()
ajf_default.fit(T_cr, E_cr, event_of_interest=2, label="Default")
ajf_default.plot_cumulative_density(ax=ax)

ax.set_xlabel("Loan Age (months)")
ax.set_ylabel("Cumulative Incidence")
ax.set_title("Competing Risks: Cumulative Incidence Functions  [100K stratified sample]")
ax.set_xlim(0, 360)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "E1_competing_risks_cif.png", bbox_inches="tight")
plt.show()

col_p = ajf_prepay.cumulative_density_.columns[0]
col_d = ajf_default.cumulative_density_.columns[0]
print(f"10-yr prepayment CIF : {ajf_prepay.cumulative_density_[col_p].loc[120]:.3f}")
print(f"10-yr default    CIF : {ajf_default.cumulative_density_[col_d].loc[120]:.4f}")

### E.1 Results — Competing Risks Interpretation

The Aalen-Johansen estimator decomposes the total exit probability into cause-specific CIFs
that sum to the overall event probability (unlike 1−KM which over-counts by ignoring
the competing event).

**Prepayment CIF** rises steeply in years 2–7, reflecting the refinancing wave driven by
rate cycles. By year 10, over 80% of loans have prepaid — mortgage borrowers are highly
rate-sensitive and rarely hold a fixed-rate loan to maturity.

**Default CIF** stays below 3% at all horizons. Despite the 2008 crisis affecting loans in
this cohort, the subordination structure and lender standards keep realised default rates low.
The asymmetry between prepayment and default risk is the central feature of mortgage credit.

**Key implication:** A model that treats default as pure censoring (standard single-risk
survival) will over-estimate the prepayment hazard, since some loans that default would
never have prepaid. Competing-risks frameworks give unbiased CIF estimates.

---
## E.2 Scenario Analysis — Interest Rate Shocks

Shock the `mortgage_rate` covariate by ±100 bp and ±200 bp and observe
the change in predicted prepayment hazard from both models:

- **Deep Cox** — reports mean log-hazard ratio `f_θ(x)` on 2,000 test loans
- **XGBoost** — reports mean monthly prepayment probability on 2,000 panel rows

In [ ]:
SHOCKS_BP = [-200, -100, 0, +100, +200]

# ── Deep Cox test set (2000 loans) ────────────────────────────────────────────
test_dc       = dc_df.iloc[n_train : n_train + 2000].copy()
base_features = scaler_dc.transform(test_dc[feat_cols_dc].fillna(test_dc[feat_cols_dc].median()))
mortgage_rate_idx = feat_cols_dc.index("mortgage_rate")

# ── XGBoost shock panel (2000 panel rows from val vintages) ───────────────────
_shock_pl = (
    pl.read_parquet(PANEL_PATH)
    .filter(
        (pl.col("vintage_year").is_between(2017, 2019)) &
        pl.col("mortgage_rate").is_not_null()
    )
    .head(4000)
)
_shock_df = _shock_pl.to_pandas()
_shock_df = pd.get_dummies(_shock_df, columns=["loan_purpose", "occupancy"], drop_first=True)
for col in FEATURES:
    if col not in _shock_df.columns:
        _shock_df[col] = 0.0
shock_base = _shock_df[FEATURES].head(2000).fillna(_shock_df[FEATURES].median())

results_scenario = {}
model_dc.eval()

for shock_bp in SHOCKS_BP:
    shocked = base_features.copy()
    shocked[:, mortgage_rate_idx] += shock_bp / 100.0

    with torch.no_grad():
        log_hz = model_dc(torch.tensor(shocked, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    results_scenario[shock_bp] = {"DeepCox_log_hz": log_hz.mean()}

    panel_sample = shock_base.copy()
    panel_sample["mortgage_rate"]  = panel_sample["mortgage_rate"] + shock_bp / 100.0
    panel_sample["rate_incentive"] = panel_sample["orig_rate"] - panel_sample["mortgage_rate"]
    xgb_proba = xgb_model.predict_proba(panel_sample[FEATURES].fillna(0))[:, 1].mean()
    results_scenario[shock_bp]["XGB_avg_proba"] = xgb_proba

# ── Plot ──────────────────────────────────────────────────────────────────────
shocks    = SHOCKS_BP
dc_scores = [results_scenario[s]["DeepCox_log_hz"] for s in shocks]
xgb_probs = [results_scenario[s]["XGB_avg_proba"]  for s in shocks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(shocks, dc_scores, "o-", color="tab:blue")
axes[0].axvline(0, ls="--", color="gray", lw=0.8)
axes[0].set_xlabel("Rate Shock (bp)")
axes[0].set_ylabel("Mean Log-Hazard Ratio")
axes[0].set_title("Deep Cox: Hazard Response to Rate Shock")
axes[0].grid(alpha=0.3)

axes[1].plot(shocks, xgb_probs, "s-", color="tab:orange")
axes[1].axvline(0, ls="--", color="gray", lw=0.8)
axes[1].set_xlabel("Rate Shock (bp)")
axes[1].set_ylabel("Avg Monthly Prepayment Probability")
axes[1].set_title("XGBoost: Prepayment Response to Rate Shock")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=3))
axes[1].grid(alpha=0.3)

plt.suptitle("Scenario Analysis: Prepayment Sensitivity to Interest Rate Shocks", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "E2_scenario_analysis.png", bbox_inches="tight")
plt.show()

print("\nScenario Summary:")
print(f"{'Shock (bp)':<12} {'DeepCox logHR':>15} {'XGB Prob':>12}")
for s in shocks:
    r = results_scenario[s]
    print(f"{s:>+12}  {r['DeepCox_log_hz']:>15.4f}  {r['XGB_avg_proba']:>12.5f}")

### E.2 Results — Scenario Analysis Interpretation

**XGBoost** shows a clear monotone response: rate cuts increase the refinancing incentive
(`rate_incentive = orig_rate − mortgage_rate`) which is the model's top feature, directly
lifting monthly prepayment probability. A −200 bp shock approximately doubles the
probability relative to a +200 bp shock.

**Deep Cox** shows a flatter response because it was trained on **static** loan features
at origination — `mortgage_rate` enters only indirectly through the macro merge.
The Cox-Time model (D.vii in main notebook) would show stronger time-varying sensitivity
since the time feature allows the hazard to respond differently at different loan ages.

**Economic interpretation:** Rate cuts are the dominant prepayment trigger.
A 200 bp cut (e.g. Fed easing cycle) can shift the portfolio's monthly prepayment
rate from ~25% to ~49% per annum — a near-doubling of turnover that significantly
shortens duration and compresses mortgage servicing income.